# Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.models as models
from torchvision.models.feature_extraction import create_feature_extractor
from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode
from sklearn.random_projection import SparseRandomProjection, johnson_lindenstrauss_min_dim


from os import path
import pandas as pd
from PIL import Image
from tqdm import tqdm
import numpy as np

from megcourselib.data.stim import load_crop

In [ ]:
PROJECT_ROOT = ".."
CHALLENGE = "challenge1"
DATA_ROOT = path.join(PROJECT_ROOT, 'data', 'brainencoding26', CHALLENGE)
STIM_ROOT = path.join(PROJECT_ROOT, 'data', 'avs_scenes')

# torchvision models

In this notebook, we will look at the basics for extracting features from neural networks in order to use them as the basis for encoding models. For this, we will first need a trained neural network. Fortunately, most well established architectures have reference implementations in PyTorch. These are available from the `torchvision` package, and usually also come with one or more sets of weights. 

This means that - to start with - we do not have to train the networks ourselves. Have a look at the [documentation](https://docs.pytorch.org/vision/main/models.html) for more details on the available models, weights, and functions.

You can also look at available models using the library directly:

In [ ]:
models.list_models()

## Choosing a model

As you can see, there are lots of models to choose from. For the purpose of this introduction we will have a closer look at ResNet18, since compared to other models it is not too large and should work well on most laptops, even without a compatible GPU. 

Next, we can look at which pretrained weights are available for this model:

In [ ]:
print(list(models.ResNet18_Weights))

ResNet18 only has weights for ImageNet 1k, so the choice is easy. Next, we load the model with the specified weights.

IMPORTANT: many models have layers that make them non deterministic during training. A common example of this is dropout, which randomly sets the activation of a subset of units in the network to 0 for every input. This is a useful regularizer during training, but should be switched off when unsing the model.

All PyTorch models have an `eval()` function that switches all of the models layers to evaluation mode. You should always call this function before extracting activations from the model (unless your research question explicitly involves introducing randomness in your predictors of course).

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.eval()

## Preprocessing

One important detail when dealing with pretrained models is that we must match any preprocessing that was applied during training of the model. Commonly, inputs are standardized to have 0 mean and unit variance for each input 'channel' (for images the input channels are the 'red', 'green' and 'blue' channels of the images). 

Additionally, since the model architectures are often designed to deal with a fixed image size we need to resize our stimuli to match the shape that is expected by the model.

In the case of models from torchvision, all of this information is made available togther with the model:

In [ ]:
original_transform = models.ResNet18_Weights.IMAGENET1K_V1.transforms()
original_transform

The above tells us that for ResNet18, input images were resized to 256 x 256 pixels, and then cropped to 224 pixels.
Inputs are then standardized with the listed mean and standard deviation.

We could use this transform directly by calling it on an image tensor like this:

`original_transform(im)`

However, since we are already doing our own cropping we build our own transform to skip this step:


In [ ]:
mean = original_transform.mean
std = original_transform.std
size = original_transform.resize_size


transform = transforms.Compose([
    # we have a numpy array, but resize needs a PIL image
    transforms.ToPILImage(),
    # directly resize to size expected by the network
    transforms.Resize(224, interpolation=InterpolationMode.BILINEAR), 
    transforms.ToTensor(),  # convert to tensor also scales to [0, 1]
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

transform

# Set up Data Loading

Load the metadata as before. We will select a single subject again to make this feasible to run on a laptop.

In [ ]:
subject = 1
metadata = pd.read_csv(path.join(DATA_ROOT, 'training', 'metadata.csv'))

subj_metadata = metadata[
    metadata.subject == subject
]

subj_metadata

The `load_crop` function is similar to the function we defined in the previous notebook. Since we will likely use it more often it is part of the `megcourselib` package and has already been imported for this notebook.

We also select a single row for testing

In [ ]:
row = subj_metadata.iloc[0]
cropsize = (224, 224)

is_valid, crop = load_crop(row.sceneID, row.mean_gx, row.mean_gy, cropsize, STIM_ROOT)

## Write a function to process a single sample

In [ ]:
def prepare_stim(row, transform):
    is_valid, crop = load_crop(row.sceneID, row.mean_gx, row.mean_gy, cropsize, STIM_ROOT)
    # apply transform
    crop_tensor = transform(crop)
    return is_valid, crop_tensor

## Testing

Load our example crop as a tensor.
As you can see, the data got converted to a torch tensor with datatype float32.
it also got reshaped to have the channel dimensions first - this is the format torch expects.

In [ ]:
is_valid, x = prepare_stim(row, transform)
print(x.shape, x.dtype, x.min(), x.max())

We can now pass the crop to our model. The output will be a 1000-dimensional vector (the number of classes the model was trained for). This output vector is usually not a good set of features for encoding but is sufficient to see how the process works.

Notice the `with torch.no_grad()` context manager. This keeps PyTorch from tracking information to compute the gradients of the computations that happen during the forward pass. These are not needed for feature extraction so we can save the memory.

Finally, models always expect a batch of data. For a single sample, we add a unitary batch dimension.

In [ ]:
with torch.no_grad():
    print('sample shape:', x.shape)
    xbatch = x[None, ...]  # adds a new axis in front
    print('sample shape with batch dimension:', xbatch.shape)
    out = model(xbatch)

print(out.shape)

# Feature Extraction

Finally, we can set up our feature extractor. Torchvision offers a utility for this, too.
We only need to know the name of the layer for which we want to extract the activation.

If you print the model, PyTorch usually shows a summary of all layers in sequence:

In [ ]:
model

For this model, the layers are nested (you can see this from the structure indicated with brackets).
You can refer to each layer with a string, by concatenating the layer names with a '.' for each layer of the hierarchy. For example:

In [ ]:
layers = ['layer1.0.conv2', 'layer4.0.conv2']

PyTorch has a `create_feature_extractor()` function to return the activation for one or more layers specified like this:

In [ ]:
extractor = create_feature_extractor(model, layers)

We now have a new model with only the layers up to the last layer from which we extract features:

In [ ]:
extractor

Call the extractor as we did with the full model:

In [ ]:
with torch.no_grad():
    feats = extractor(xbatch)

This returns a dictionary with the features for the layers we have requested.

In [ ]:
for k, v in feats.items():
    print(f"{k}: {v.shape}")

# Extract features for all specified layers

Below we build a simple pipeline to extract features for all fixations of a single subject.
As with the example in the previous notebook, this code should serve as an example of the 'ingredients' for feature extraction from a neural network and is not necessarily the ideal way to do this efficiently.

Note: activations from neural networks can be very high dimensional, especially for the early convolutional layers in many networks. In the implementation below, the activations from a convolutional layer that have shape (batch, channels, height, width) are averaged in space (over the last two axes). This significantly reduces the size of embeddings, but of course also discards some information. For your projects it is worth thinking about whether this is what you want.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class StimDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        is_valid, crop_tensor = prepare_stim(row, self.transform)
        return is_valid, crop_tensor


def extract_features(dataloader, extractor, device="cpu"):
    """
    device - set 'cuda' or 'mps' for nvidia or mac gpu if available. Ensure the model is on the same device
    """
    extractor.eval()
    features = {}
    projectors = {}  # one fitted projector per layer

    mask = []  # store True for crops that were extracted, False if not

    with torch.no_grad():
        for is_valid, batch in tqdm(dataloader):
            mask.extend(is_valid.tolist())
            batch = batch.to(device)  # move to device
            out = extractor(batch)
           
            for layer, tensor in out.items():
                arr = tensor.cpu().numpy()

                # for conv layers, average over space
                if len(arr.shape) == 4:
                    flat = arr.mean(axis=(-1, -2))  # avg over space
                else:
                    flat = arr
 
                # # fit projector on first batch only, reuse for subsequent batches
                # if layer not in projectors:
                #     projectors[layer] = SparseRandomProjection(n_components=jl_n_components)
                #     projectors[layer].fit(flat)

                # projected = projectors[layer].transform(flat)
                
                if layer not in features:
                    features[layer] = []
                features[layer].append(flat)

    return {layer: np.concatenate(arrays) for layer, arrays in features.items()}, mask



In [ ]:
DEVICE = 'mps'  # or 'cpu' or 'cuda' . mps is available on macbooks with M1-5 processors. Cuda if you have an nvidia gpu. cpu will always work but can be slower
jl_eps=0.3  # increase for projection to smaller space if needed
batch_size = 256  # larger is potentially faster (especially on gpu), but requires more RAM

# jl_n_components = johnson_lindenstrauss_min_dim(len(subj_metadata), eps=jl_eps)

model.to(DEVICE)  # move model to device
dataset = StimDataset(subj_metadata, transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)  # IMPORTANT: do not shuffle. Adjust batch size as needed to fit into RAM

# run feature extraction
all_features, valid_mask = extract_features(dataloader, extractor, device=DEVICE)

# drop metadata for failed crops
meta = subj_metadata.loc[valid_mask].reset_index()

# drop features for failed crops (these are features where the input was all zeros)
all_features = {k: features[valid_mask] for k, features in all_features.items()}

# Build an encoding model

We build an encoding model 

In [ ]:
for k, v in all_features.items():
    print(f"{k}: {v.shape}")

print()
print(f"metadata: {meta.shape}")

In [ ]:
# select a layer for the encoding model
encoding_layer = "layer4.0.conv2"

In [ ]:
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.feature_selection import VarianceThreshold

# load meg data
meg_train = np.load(path.join(DATA_ROOT, 'training', 'meg_110ms.npy'))

# get data to match the indices for which we just extracted features
meg = meg_train[subj_metadata.index]
meg = meg[valid_mask]

# standardize the meg data
scaler_meg = RobustScaler().fit(meg)
meg = scaler_meg.transform(meg)

pred = all_features[encoding_layer].astype(np.double)


# neural networks with ReLU activations can have 'dead neurons' that are never activated. This causes issues with standardization
# drop these dimensions
selector = VarianceThreshold(threshold=1e-5)
pred = selector.fit_transform(pred)


print()
print()
print('========================== DATA FOR ENCODING MODEL ==========================')
print('MEG data:', meg.shape, 'predictors:', pred.shape, 'metadata:', meta.shape)


In [ ]:
print("pred constant cols:", (pred.std(axis=0) < 1e-10).sum())
print("neural data constant cols:", (meg.std(axis=0) < 1e-10).sum())

# Train the encoding model

In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold
from sklearn.metrics import explained_variance_score
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA


nfold = 5
alphas = np.logspace(-3, 16, 128)
kf = KFold(n_splits=nfold, shuffle=True, random_state=42)
n_trials, n_chan = meg.shape

pipeline = Pipeline([
    ('pca', PCA(n_components=.99)),
    # ('scaler', StandardScaler()),
    ('ridge', RidgeCV(alphas=alphas, alpha_per_target=True))
])

def correlation_score(y_true, y_pred):
    return np.array([
        np.corrcoef(y_true[:, c], y_pred[:, c])[0, 1]
        for c in range(y_true.shape[1])
    ])


r_per_fold  = np.empty((nfold, n_chan))
for i, (train_idx, test_idx) in enumerate(kf.split(pred)):
    print(f">>> fold {i + 1} / {nfold}")

    X_train, X_test = pred[train_idx], pred[test_idx]
    y_train, y_test = meg[train_idx], meg[test_idx]

    model = pipeline.fit(X_train, y_train)
    print(f"PCA components retained: {pipeline.named_steps['pca'].n_components_}")

    predictions = pipeline.predict(X_test)

    r_per_fold[i]  = correlation_score(y_test, predictions)

print(f"r:  mean={r_per_fold.mean():.5f}, worst sensor={r_per_fold.mean(axis=0).min():.5f}  best sensor={r_per_fold.mean(axis=0).max():.5f}")


In [ ]:
import mne
import matplotlib.pyplot as plt
# show on topoplot

# get information about sensor location for the subject
fif_path = path.join(DATA_ROOT, 'training', f"sub-{str(subject).zfill(2)}_grad_info.fif")
info = mne.io.read_info(fif_path)

# plot data
fig, ax = plt.subplots(figsize=(6, 6))
im, _ = mne.viz.plot_topomap(
    data=r_per_fold.mean(axis=0),
    pos=info,
    ch_type='grad',
    axes=ax,
    show=False,          # prevent auto-display so we can add the colorbar
    vlim=(r_per_fold.mean(axis=0).min(), r_per_fold.mean(axis=0).max())
)

plt.colorbar(im, ax=ax, shrink=0.7, label='pearson correlation')
plt.tight_layout()
plt.show()
